In [ ]:
%pwd

In [ ]:
print("ok")

In [ ]:
import os
os.chdir("../")

In [ ]:
%pwd

In [ ]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [ ]:
pip install langchain-text-splitters


In [ ]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [ ]:
def load_pdf_file(data):
    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )
    documents = loader.load()
    return documents

In [ ]:
extracted_data= load_pdf_file(data='Data/')

In [ ]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader

def load_pdf_file(data):
    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )
    documents = loader.load()
    return documents

In [ ]:
import os

# Create the data directory inside 'Medical-chatbot-genAi'
os.makedirs("data", exist_ok=True)
print("Data directory created successfully. Please place your PDF files inside it.")

In [ ]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

def load_pdf_file(data_path):
    loader = DirectoryLoader(
        data_path,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )
    documents = loader.load()
    return documents

# Call the function with the matching folder name
extracted_data = load_pdf_file(data_path='data/')
print(f"Successfully loaded {len(extracted_data)} pages/documents.")


In [ ]:
# Change 'data/' to '../data/' to step out of the research folder
extracted_data = load_pdf_file(data_path='../data/')
print(f"Successfully loaded {len(extracted_data)} pages/documents.")

In [ ]:
#extracted_data

In [ ]:
def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    text_chunks = text_splitter.split_documents(extracted_data)
    return text_chunks

In [ ]:
text_chunks = text_split(extracted_data)
print("Length of Text Chunks:", len(text_chunks))

In [ ]:
text_chunks

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

def download_hugging_face_embeddings():
    embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
    return embeddings

embeddings = download_hugging_face_embeddings()

In [ ]:
#query_result = embeddings.embed_query("Hello world")
print("Length", len(query_result))

In [ ]:
import os
from pinecone.grpc import PineconeGRPC as Pinecone
from pinecone import ServerlessSpec

# 1. Initialize Pinecone client connection
pc = Pinecone(api_key="pcsk_5EM81u_88V4e3B9chnLx3kY1qEfnm89UuJRRPDVSFtaM74h59vnaNYSMJxL2PxD5B8v3ox")

index_name = "medicalbot"

# 2. Check if index already exists before creating to prevent overwrite errors
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384,          # Matches all-MiniLM-L6-v2 vector output width
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    print(f"Successfully created Pinecone index: '{index_name}'")
else:
    print(f"Index '{index_name}' already exists.")

# 3. Target the active index to prepare for document ingestion
index = pc.Index(index_name)

In [ ]:

import os
from dotenv import load_dotenv

   
load_dotenv()


api_key = os.environ.get("PINECONE_API_KEY")


In [ ]:
import os
import time
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from pinecone import Pinecone, ServerlessSpec

print("--- Starting Clean Native Pipeline ---")

# 1. System Credentials & Environmental Setup
index_name = "testbot"
api_key = "pcsk_5EM81u_88V4e3B9chnLx3kY1qEfnm89UuJRRPDVSFtaM74h59vnaNYSMJxL2PxD5B8v3ox"
os.environ["PINECONE_API_KEY"] = api_key

# 2. Extract Text from PDF Source Documents
print("Step 1: Extracting text from PDF data source...")
loader = DirectoryLoader(
    "../data/",
    glob="*.pdf",
    loader_cls=PyPDFLoader
)
extracted_data = loader.load()
print(f"-> Successfully extracted {len(extracted_data)} total pages.")

# 3. Text Chunk Segmentation Frame Creation
print("Step 2: Splitting text document frames into semantic chunks...")
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
text_chunks = text_splitter.split_documents(extracted_data)
print(f"-> Total text chunks generated: {len(text_chunks)}")

# 4. Truncate Data Scope safely to stay inside Free Tier limits
sampled_chunks = text_chunks[:5000]
print(f"-> Selected first {len(sampled_chunks)} chunks for streaming optimization.")

# 5. Initialize Sentence-Transformers Local Model
print("Step 3: Initializing embedding transformer client engine...")
embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

# 6. Check and Automatically Create Index if Missing
print("Step 4: Connecting to Pinecone Cloud Database...")
pc = Pinecone(api_key=api_key)

# Check existing indexes using the modern v6.x SDK syntax
existing_indexes = [idx['name'] for idx in pc.list_indexes()]

if index_name not in existing_indexes:
    print(f"-> Index '{index_name}' not found. Creating it automatically...")
    pc.create_index(
        name=index_name,
        dimension=384, # Dimensions for all-MiniLM-L6-v2
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1" # Free tier default region
        )
    )
    # Wait for the cloud index to provision completely
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(1)
    print("-> Index created and ready!")
else:
    print(f"-> Found existing index '{index_name}'.")

# 7. Direct Batch Upsert Using Native Client
print("Step 5: Vectorizing and streaming chunks directly via Native Pinecone SDK...")
index = pc.Index(index_name)

# Extract text strings and calculate embeddings
raw_texts = [doc.page_content for doc in sampled_chunks]
vectors_list = embeddings.embed_documents(raw_texts)

# Format payload to precisely match what LangChain requires for downstream RAG retrieval
upsert_payload = []
for i, (text, vector, original_doc) in enumerate(zip(raw_texts, vectors_list, sampled_chunks)):
    metadata = original_doc.metadata.copy()
    metadata["text"] = text  # Critical for future retriever lookups
    upsert_payload.append((f"id_{i}", vector, metadata))

# Stream batches directly to the cluster endpoint
batch_size = 100
for idx in range(0, len(upsert_payload), batch_size):
    batch = upsert_payload[idx:idx + batch_size]
    index.upsert(vectors=batch)

print("\n🚀 --- Pipeline Complete! Pinecone Vector Store initialized successfully! ---")

In [ ]:
import os
from langchain_pinecone import PineconeVectorStore
from langchain_huggingface import HuggingFaceEmbeddings

# 1. Setup credentials
index_name = "testbot"
os.environ["PINECONE_API_KEY"] = "pcsk_5EM81u_88V4e3B9chnLx3kY1qEfnm89UuJRRPDVSFtaM74h59vnaNYSMJxL2PxD5B8v3ox"

# 2. Initialize the exact same embedding model used during ingestion
print("Loading embedding model...")
embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

# 3. Load the existing index from the Pinecone cloud
print(f"Connecting to existing Pinecone index: '{index_name}'...")
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name, 
    embedding=embeddings
)

print("Index successfully loaded into memory!")

In [ ]:
# 1. Convert your vector store wrapper into a search retriever
retriever = docsearch.as_retriever(search_kwargs={"k": 3})

# 2. Define a medical test query
query = "What are the primary symptoms of Acne?"

# 3. Fetch matched content blocks from your PDF text layers
print(f"Searching index for: '{query}'...")
matching_docs = retriever.invoke(query)

# 4. Print the text chunks returned by Pinecone
print(f"\nRetrieved {len(matching_docs)} relevant context blocks:")
for i, doc in enumerate(matching_docs):
    print(f"\n--- Context Block {i+1} (Page {doc.metadata.get('page', 'Unknown')}) ---")
    print(doc.page_content.strip()[:400] + "...")

In [ ]:
%pip install --force-reinstall langchain langchain-core langchain-community

In [ ]:
%pip install -U langchain


In [ ]:
import os
from langchain_pinecone import PineconeVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

print("--- Starting Full Self-Contained Gemini RAG Generation ---")

# 1. Environment Credentials Configuration
index_name = "testbot"
os.environ["PINECONE_API_KEY"] = "pcsk_5EM81u_88V4e3B9chnLx3kY1qEfnm89UuJRRPDVSFtaM74h59vnaNYSMJxL2PxD5B8v3ox"

# Set your active Google AI Studio Key here
os.environ["GEMINI_API_KEY"] = "AQ.Ab8RN6Jw4Qu7ud3leyhX0TzI2MQxw-nV9yLOUSB8DKmmDklE8g"

# 2. Re-establish Vector Database Connections
print("1/4: Re-initializing embedding transformer models...")
embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

print(f"2/4: Loading active cloud index instance pointers for '{index_name}'...")
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name, 
    embedding=embeddings
)
retriever = docsearch.as_retriever(search_kwargs={"k": 3})

# 3. Load Gemini Generation Engine (Updated to the modern Gemini 2.5 architecture)
print("3/4: Binding Gemini chat completion wrappers...")
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", 
    temperature=0.4, 
    max_output_tokens=500
)

# 4. Define Clinical Instructions System Schema Layout
system_prompt = (
    "You are a helpful and professional medical assistant. Use the following pieces of "
    "retrieved context to answer the user's question accurately.\n"
    "If you do not know the answer, or if the context doesn't contain it, say clearly that you "
    "do not know. Do not try to make up an answer.\n\n"
    "Retrieved Context:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

print("4/4: Execution pipeline connections ready!")

# 5. Pipeline Execution Logic Block
try:
    test_query = "What causes Acne?"
    print(f"\n-> Fetching textbook text frames from Pinecone cluster for: '{test_query}'...")
    
    # Query your active retriever securely
    matching_docs = retriever.invoke(test_query)
    combined_context = "\n\n".join([doc.page_content for doc in matching_docs])
    
    print("-> Streaming context frames payload to Gemini system instances...")
    formatted_messages = prompt.format_messages(context=combined_context, input=test_query)
    
    # Execute end-to-end model query transformation string
    response = llm.invoke(formatted_messages)
    
    print("\n" + "="*50)
    print("🩺 MEDICAL CHATBOT OUTPUT ANSWER (GEMINI):")
    print("="*50)
    print(response.content)
    print("="*50)
    
except Exception as e:
    print(f"\nAn error occurred during generation execution processing: {str(e)}")